# 02 - Train, Test, and Validation Split

In this notebook, we will properly split our data into three sets to prevent the data leakage issues we saw in Notebook 1.

## Concept Overview
* **Train Set**: The model learns from this.
* **Validation Set**: We (the humans) use this to tune hyperparameters.
* **Test Set**: Kept in a vault until the very end to get an unbiased generalization score.

## Scikit-Learn Implementation
We will use Scikit-Learn's `train_test_split` to create these three sets.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np

# Generate 1000 samples
X, y = make_classification(n_samples=1000, n_features=10, n_classes=2, random_state=42)

# 1. Split into Train (80%) and Test (20%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 2. Split the Temp set into Train (75% of temp) and Validation (25% of temp)
# This results in a final split of 60% Train, 20% Val, 20% Test
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

print(f'Training Set: {len(X_train)} samples')
print(f'Validation Set: {len(X_val)} samples')
print(f'Test Set: {len(X_test)} samples')

## Stratified Splitting
If our dataset is highly imbalanced, a random split might accidentally put all the minority class samples into the training set. We use `stratify=y` to prevent this.

In [ ]:
# Imbalanced dataset (95% class 0, 5% class 1)
X_imb, y_imb = make_classification(n_samples=1000, weights=[0.95, 0.05], random_state=42)

# Bad Split (Random)
X_t_bad, X_test_bad, y_t_bad, y_test_bad = train_test_split(X_imb, y_imb, test_size=0.2)
print('Random Split Test Set Minorities:', sum(y_test_bad))

# Good Split (Stratified)
X_t_good, X_test_good, y_t_good, y_test_good = train_test_split(X_imb, y_imb, test_size=0.2, stratify=y_imb)
print('Stratified Split Test Set Minorities:', sum(y_test_good))

## Time Series Split Demonstration
Never shuffle time series data. Train on the past, test on the future.

In [ ]:
import pandas as pd

dates = pd.date_range(start='2020-01-01', periods=100, freq='D')
df = pd.DataFrame({'Date': dates, 'Value': np.random.randn(100)})

# Chronological Split (Train on first 80 days, Test on last 20)
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

print(f'Train ends: {train_df["Date"].iloc[-1].date()}')
print(f'Test begins: {test_df["Date"].iloc[0].date()}')

## Industry Notes
The Test set is sacred. If you ever make a decision about your model (changing a parameter, adding a feature) based on its performance on the Test set, you have invalidated your evaluation.

## Exercises
1. Write a function that automatically takes a DataFrame and returns Train/Val/Test dataframes given a list of percentages like `[0.7, 0.15, 0.15]`.
2. Implement a Scikit-Learn Pipeline that scales the data *after* splitting to prevent target leakage.